In [43]:
import torch
import torch.nn as nn
import sys
import numpy as np
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # must come BEFORE any src import

from src.model import MatrixFactorization
from src.metrics import evaluate
from src.data import load_ratings, time_split, build_id_maps
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

ratings = load_ratings()
train, test_warm = time_split(ratings)
user_to_idx, movie_to_idx = build_id_maps(train)
n_users, n_movies = len(user_to_idx), len(movie_to_idx)

print(f"n_users: {n_users}, n_movies: {n_movies}")

n_users: 5400, n_movies: 3662


In [44]:
train_u = train["user_id"].map(user_to_idx).values
train_m = train["movie_id"].map(movie_to_idx).values
train_r = train["rating"].values

seen_by_user = {}
for u, m in zip(train_u, train_m):
    seen_by_user.setdefault(u, set()).add(m)

movie_counts = np.zeros(n_movies)
for m in train_m:
    movie_counts[m] += 1

# movie_counts = movie_counts ** 0.75

# movie_probs = movie_counts / movie_counts.sum()  
# def sample_negative(user_idx: int) -> int:
#     """Draw a popularity-weighted random movie this user has NOT interacted with."""
#     seen = seen_by_user[user_idx]
#     while True:
#         random_movie = np.random.choice(n_movies, p=movie_probs)
#         if random_movie not in seen:
#             return random_movie
def sample_negative(user_idx: int) -> int:
    """Draw a UNIFORM random movie this user has NOT interacted with."""
    seen = seen_by_user[user_idx]
    while True:
        random_movie = np.random.randint(n_movies)
        if random_movie not in seen:
            return random_movie

In [45]:
def make_batch(batch_indices, alpha=40):
    """Given row indices into the training data, build a batch of
    positives and sampled negatives.

    Returns (users, movies, labels) as tensors.
    """
    users = train_u[batch_indices]     
    pos_movies = train_m[batch_indices]
    ratings = train_r[batch_indices]

    neg_movies = np.array([sample_negative(u) for u in users])

    batch_users = np.concatenate([users, users])  
    batch_movies = np.concatenate([pos_movies, neg_movies])
    batch_labels = np.concatenate([np.ones(len(users)), np.zeros(len(users))])
    batch_confidence = np.concatenate([1 + alpha * ratings, np.ones(len(users))])

    batch_users = torch.tensor(batch_users, dtype=torch.long)
    batch_movies = torch.tensor(batch_movies, dtype=torch.long)
    batch_labels = torch.tensor(batch_labels, dtype=torch.float)
    batch_confidence = torch.tensor(batch_confidence, dtype=torch.float)

    return batch_users, batch_movies, batch_labels, batch_confidence

In [46]:
bu, bm, bl, bc = make_batch(np.array([0, 1, 2]))
print(bu.shape, bm.shape, bl.shape, bc.shape)  # all (6,)
print(bc)  # first 3 are 1+40*rating, last 3 are 1.0

torch.Size([6]) torch.Size([6]) torch.Size([6]) torch.Size([6])
tensor([161., 161., 201.,   1.,   1.,   1.])


In [47]:
def wmf_loss(preds, labels, confidence):
    """Confidence-weighted squared error (WMF objective).

    Args:
        preds: model's raw dot-product outputs, shape (2B,)
        labels: 0/1 preference targets, shape (2B,)
        confidence: per-example confidence weights, shape (2B,)

    Returns:
        scalar loss
    """
    # 1. squared error per example: (labels - preds) squared
    squared_error = (labels - preds) ** 2

    # 2. weight each example's error by its confidence
    weighted = confidence * squared_error

    # 3. reduce to a single scalar (sum or mean — you decide
    return weighted.mean()

In [48]:
model = MatrixFactorization(n_users, n_movies, k=50)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)

n_epochs = 5
batch_size = 1024
n_train = len(train_u)

for epoch in range(n_epochs):
    perm = np.random.permutation(n_train) 

    total_loss = 0.0
    for i in range(0, n_train, batch_size):
        batch_indices = perm[i:i + batch_size]

        users, movies, labels, confidence = make_batch(batch_indices)

        preds = model(users, movies)
        loss = wmf_loss(preds, labels, confidence)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_indices)

    avg_loss = total_loss / n_train
    print(f"Epoch {epoch+1}: WMF loss = {avg_loss:.4f}")

Epoch 1: WMF loss = 1321.8205
Epoch 2: WMF loss = 143.2931
Epoch 3: WMF loss = 38.9051
Epoch 4: WMF loss = 15.4244
Epoch 5: WMF loss = 8.3551


In [49]:
model.eval()
idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}
all_movie_idx = torch.arange(n_movies)

def recommend_mf(user_idx, k=10):
    with torch.no_grad():
        users = torch.full((n_movies,), user_idx, dtype=torch.long)
        scores = model(users, all_movie_idx)
    ranked = torch.argsort(scores, descending=True).tolist()
    seen = seen_by_user.get(user_idx, set())
    return [idx_to_movie[i] for i in ranked if i not in seen][:k]

warm_user_ids = list(test_warm["user_id"].unique())
recs_wmf = {uid: recommend_mf(user_to_idx[uid], k=10) for uid in warm_user_ids}

relevant_ratings = test_warm[test_warm["rating"] >= 4]
relevant_by_user = relevant_ratings.groupby("user_id")["movie_id"].apply(set).to_dict()

results_wmf = evaluate(recs_wmf, relevant_by_user, k=10)
print(f"Precision@10: {results_wmf['precision']:.4f}")
print(f"Recall@10:    {results_wmf['recall']:.4f}")
print(f"NDCG@10:      {results_wmf['ndcg']:.4f}")

Precision@10: 0.0021
Recall@10:    0.0002
NDCG@10:      0.0026


In [50]:
# pick one warm user, look at their top scores
uid = warm_user_ids[0]
user_idx = user_to_idx[uid]
with torch.no_grad():
    users = torch.full((n_movies,), user_idx, dtype=torch.long)
    scores = model(users, all_movie_idx)

print("Score range:", scores.min().item(), "to", scores.max().item())
print("Top recommended:", recommend_mf(user_idx, k=10))

Score range: -7.432770252227783 to 9.457208633422852
Top recommended: [np.int64(3796), np.int64(958), np.int64(3553), np.int64(3045), np.int64(2675), np.int64(3241), np.int64(2317), np.int64(1582), np.int64(2847), np.int64(2626)]


In [51]:
for uid in warm_user_ids:
    if uid in relevant_by_user and len(relevant_by_user[uid]) > 0:
        print("User:", uid)
        print("Relevant:", sorted(relevant_by_user[uid])[:10])
        print("Recommended:", recs_wmf[uid])
        print("Overlap:", set(recs_wmf[uid]) & relevant_by_user[uid])
        break

User: 1875
Relevant: [10, 11, 260, 329, 539, 648, 674, 736, 750, 780]
Recommended: [np.int64(3796), np.int64(958), np.int64(3553), np.int64(3045), np.int64(2675), np.int64(3241), np.int64(2317), np.int64(1582), np.int64(2847), np.int64(2626)]
Overlap: set()


In [52]:
counts = train["movie_id"].value_counts()
print("Train counts for WMF recs:", [int(counts.get(m, 0)) for m in recs_wmf[1875]])

Train counts for WMF recs: [4, 11, 43, 93, 7, 34, 6, 48, 25, 23]


In [53]:
uid = 1875
user_idx = user_to_idx[uid]

with torch.no_grad():
    users = torch.full((n_movies,), user_idx, dtype=torch.long)
    scores = model(users, all_movie_idx)

order = torch.argsort(scores, descending=True).tolist()
rank_of = {movie_idx: rank for rank, movie_idx in enumerate(order)}

liked_original = relevant_by_user[uid]
liked_dense = [movie_to_idx[m] for m in liked_original if m in movie_to_idx]
ranks = sorted(rank_of[d] for d in liked_dense)
print("Ranks of liked movies (out of", n_movies, "):", ranks[:20])

Ranks of liked movies (out of 3662 ): [421, 436, 517, 539, 624, 704, 754, 980, 1095, 1198, 1235, 1247, 1248, 1341, 1355, 1421, 1501, 1511, 1544, 1547]


In [54]:
# how big are the learned factor vectors vs the biases?
uf = model.user_factors.weight.detach()
mf = model.movie_factors.weight.detach()
ub = model.user_bias.weight.detach()
mb = model.movie_bias.weight.detach()

print("user factors  - mean abs:", uf.abs().mean().item())
print("movie factors - mean abs:", mf.abs().mean().item())
print("user bias     - mean abs:", ub.abs().mean().item())
print("movie bias    - mean abs:", mb.abs().mean().item())
print("global bias:", model.global_bias.item())

user factors  - mean abs: 0.4364347755908966
movie factors - mean abs: 0.3577561676502228
user bias     - mean abs: 0.13473108410835266
movie bias    - mean abs: 0.2023962140083313
global bias: 0.8834547996520996


In [55]:
uid = 1875
user_idx = user_to_idx[uid]
with torch.no_grad():
    users = torch.full((n_movies,), user_idx, dtype=torch.long)
    scores = model(users, all_movie_idx)
order = torch.argsort(scores, descending=True).tolist()

seen = seen_by_user[user_idx]
top20 = order[:20]
print("Of top-20 scored movies, how many are TRAIN-seen:",
      sum(1 for m in top20 if m in seen), "/ 20")

Of top-20 scored movies, how many are TRAIN-seen: 0 / 20


In [56]:
u1 = user_to_idx[warm_user_ids[0]]
u2 = user_to_idx[warm_user_ids[30]]

recs1 = recommend_mf(u1, k=20)
recs2 = recommend_mf(u2, k=20)

print("User 1 top 20:", recs1)
print("User 2 top 20:", recs2)
print("Overlap:", len(set(recs1) & set(recs2)), "/ 20")

User 1 top 20: [np.int64(3796), np.int64(958), np.int64(3553), np.int64(3045), np.int64(2675), np.int64(3241), np.int64(2317), np.int64(1582), np.int64(2847), np.int64(2626), np.int64(3903), np.int64(1174), np.int64(3401), np.int64(2898), np.int64(3787), np.int64(2931), np.int64(3240), np.int64(3678), np.int64(3738), np.int64(522)]
User 2 top 20: [np.int64(1447), np.int64(958), np.int64(3222), np.int64(877), np.int64(1489), np.int64(1943), np.int64(525), np.int64(970), np.int64(3281), np.int64(2077), np.int64(662), np.int64(1170), np.int64(893), np.int64(1421), np.int64(726), np.int64(116), np.int64(3106), np.int64(1727), np.int64(2187), np.int64(3031)]
Overlap: 1 / 20


In [57]:

counts = train["movie_id"].value_counts()
liked = relevant_by_user[1875]

popular_liked = [m for m in liked if counts.get(m, 0) > 200 and m in movie_to_idx]
niche_liked   = [m for m in liked if counts.get(m, 0) <= 200 and m in movie_to_idx]

pop_ranks   = sorted(rank_of[movie_to_idx[m]] for m in popular_liked)
niche_ranks = sorted(rank_of[movie_to_idx[m]] for m in niche_liked)

print("Popular liked movies, ranks:", pop_ranks[:15])
print("Niche liked movies, ranks:  ", niche_ranks[:15])

Popular liked movies, ranks: [436, 517, 539, 704, 980, 1095, 1198, 1235, 1341, 1355, 1421, 1501, 1511, 1544, 1547]
Niche liked movies, ranks:   [421, 624, 754, 1247, 1248, 2319, 3137]


In [58]:
# for each warm user, get their most-recent liked (>=4) test item
test_liked = test_warm[test_warm["rating"] >= 4].copy()

# sort by time so the last entry per user is their most recent
test_liked = test_liked.sort_values("timestamp")

# FILL: for each user, keep only their most-recent liked item
#   hint: groupby user_id, then take the LAST row per group (since sorted by time)
held_out = test_liked.groupby("user_id").last()

print(held_out.shape)
print(held_out.head())

(1122, 3)
         movie_id  rating   timestamp
user_id                              
635          3507       4   979143120
639          2411       4   990329284
641           265       4  1019443358
646             1       5   975782835
648          2612       4  1044744417


In [59]:
N_NEG = 100  # number of random distractors per user

def evaluate_loo(user_id, true_movie, k=10, n_neg=N_NEG):
    """Leave-one-out: rank the true held-out item against n_neg random unseen items.
    Returns (hit, ndcg) for this user."""
    user_idx = user_to_idx[user_id]
    true_idx = movie_to_idx[true_movie]
    seen = seen_by_user[user_idx]

    # 1. sample n_neg random movie indices the user hasn't seen (and != true item)
    negatives = []
    while len(negatives) < n_neg:
        cand = np.random.randint(n_movies)
        if cand not in seen and cand != true_idx:
            negatives.append(cand)

    # 2. candidate set = true item + negatives
    candidates = [true_idx] + negatives

    # 3. score all candidates with the model
    with torch.no_grad():
        u = torch.full((len(candidates),), user_idx, dtype=torch.long)
        m = torch.tensor(candidates, dtype=torch.long)
        scores = model(u, m)

    # 4. find the RANK of the true item among candidates
    #    FILL: the true item is at position 0 in `candidates`.
    #    sort candidate positions by score descending; find where position 0 lands
    order = torch.argsort(scores, descending=True).tolist()
    rank = order.index(0)
    # 5. hit@k and ndcg@k
    hit = 1.0 if rank < k else 0.0
    ndcg = 1.0 / np.log2(rank + 2) if rank < k else 0.0
    return hit, ndcg



In [60]:
hits, ndcgs = [], []
for user_id in held_out.index:
    true_movie = held_out.loc[user_id, "movie_id"]
    if true_movie not in movie_to_idx:   # skip if the held-out item isn't in train space
        continue
    hit, ndcg = evaluate_loo(user_id, true_movie, k=10)
    hits.append(hit)
    ndcgs.append(ndcg)

print(f"Leave-one-out over {len(hits)} users:")
print(f"Hit Rate@10: {np.mean(hits):.4f}")
print(f"NDCG@10:     {np.mean(ndcgs):.4f}")

Leave-one-out over 1116 users:
Hit Rate@10: 0.0376
NDCG@10:     0.0183


In [61]:
def train_wmf(k=50, lr=0.01, weight_decay=1e-5, n_epochs=5, batch_size=1024, verbose=False):
    """Train a WMF model and return it."""
    model = MatrixFactorization(n_users, n_movies, k=k)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_train = len(train_u)

    for epoch in range(n_epochs):
        perm = np.random.permutation(n_train)
        for i in range(0, n_train, batch_size):
            batch_indices = perm[i:i + batch_size]
            users, movies, labels, confidence = make_batch(batch_indices)
            preds = model(users, movies)
            loss = wmf_loss(preds, labels, confidence)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose:
            print(f"  epoch {epoch+1}: loss={loss.item():.2f}")
    return model


def score_loo(model, k=10):
    """Run leave-one-out eval, return (hit_rate, ndcg)."""
    model.eval()
    hits, ndcgs = [], []
    for user_id in held_out.index:
        true_movie = held_out.loc[user_id, "movie_id"]
        if true_movie not in movie_to_idx:
            continue
        hit, ndcg = evaluate_loo(user_id, true_movie, k=k)
        hits.append(hit)
        ndcgs.append(ndcg)
    return np.mean(hits), np.mean(ndcgs)

In [40]:
m = train_wmf(n_epochs=5, verbose=True)   # watch the loss drop
print("Training done. Now evaluating THIS model.\n")

# evaluate m explicitly — score it directly, not via a global
m.eval()
hits, ndcgs = [], []
for user_id in held_out.index:
    true_movie = held_out.loc[user_id, "movie_id"]
    if true_movie not in movie_to_idx:
        continue
    user_idx = user_to_idx[user_id]
    true_idx = movie_to_idx[true_movie]
    seen = seen_by_user[user_idx]
    negs = []
    while len(negs) < 100:
        c = np.random.randint(n_movies)
        if c not in seen and c != true_idx:
            negs.append(c)
    cands = [true_idx] + negs
    with torch.no_grad():
        u = torch.full((len(cands),), user_idx, dtype=torch.long)
        mov = torch.tensor(cands, dtype=torch.long)
        scores = m(u, mov)
    rank = torch.argsort(scores, descending=True).tolist().index(0)
    hits.append(1.0 if rank < 10 else 0.0)
    ndcgs.append(1.0/np.log2(rank+2) if rank < 10 else 0.0)

print(f"Hit@10: {np.mean(hits):.4f}, NDCG@10: {np.mean(ndcgs):.4f}")

  epoch 1: loss=367.32
  epoch 2: loss=86.05
  epoch 3: loss=31.65
  epoch 4: loss=16.06
  epoch 5: loss=6.41
Training done. Now evaluating THIS model.

Hit@10: 0.0296, NDCG@10: 0.0140


In [41]:
# draw 1000 negatives for user 0, see if they skew popular
test_negs = [sample_negative(0) for _ in range(1000)]
counts = train["movie_id"].value_counts()
# map dense back to original to check popularity
idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}
avg_pop = np.mean([counts.get(idx_to_movie[n], 0) for n in test_negs])
print("Avg popularity of sampled negatives:", avg_pop)

Avg popularity of sampled negatives: 183.01


In [65]:
trained_model = train_wmf(n_epochs=40)

trained_model.eval()
ranks = []
for user_id in list(held_out.index)[:200]:
    tm = held_out.loc[user_id, "movie_id"]
    if tm not in movie_to_idx:
        continue
    ui, ti = user_to_idx[user_id], movie_to_idx[tm]
    seen = seen_by_user[ui]
    negs = []
    while len(negs) < 100:
        c = np.random.randint(n_movies)
        if c not in seen and c != ti:
            negs.append(c)
    cands = [ti] + negs
    with torch.no_grad():
        user_tensor = torch.full((len(cands),), ui, dtype=torch.long)
        movie_tensor = torch.tensor(cands, dtype=torch.long)
        scores = trained_model(user_tensor, movie_tensor)
    ranks.append(torch.argsort(scores, descending=True).tolist().index(0))

print("Median rank after 40 epochs:", np.median(ranks))
print("Mean rank:", np.mean(ranks))

Median rank after 40 epochs: 32.0
Mean rank: 33.58585858585859


In [63]:
# fully self-contained sanity test
import numpy as np, torch

# confirm uniform sampler
def sample_neg_uniform(user_idx):
    seen = seen_by_user[user_idx]
    while True:
        c = np.random.randint(n_movies)
        if c not in seen:
            return c

# quick check: are seen_by_user keys dense indices?
print("Sample seen_by_user key:", list(seen_by_user.keys())[:3])
print("n_users:", n_users, "n_movies:", n_movies)
print("Sample held_out user:", held_out.index[0], "→ dense:", user_to_idx.get(held_out.index[0]))

Sample seen_by_user key: [np.int64(0), np.int64(1), np.int64(2)]
n_users: 5400 n_movies: 3662
Sample held_out user: 635 → dense: 5399


In [66]:
trained_model.eval()
hits, ndcgs = [], []
for user_id in held_out.index:
    tm = held_out.loc[user_id, "movie_id"]
    if tm not in movie_to_idx:
        continue
    ui, ti = user_to_idx[user_id], movie_to_idx[tm]
    seen = seen_by_user[ui]
    negs = []
    while len(negs) < 100:
        c = np.random.randint(n_movies)
        if c not in seen and c != ti:
            negs.append(c)
    cands = [ti] + negs
    with torch.no_grad():
        user_tensor = torch.full((len(cands),), ui, dtype=torch.long)
        movie_tensor = torch.tensor(cands, dtype=torch.long)
        scores = trained_model(user_tensor, movie_tensor)
    rank = torch.argsort(scores, descending=True).tolist().index(0)
    hits.append(1.0 if rank < 10 else 0.0)
    ndcgs.append(1.0/np.log2(rank+2) if rank < 10 else 0.0)

print(f"Hit@10: {np.mean(hits):.4f}, NDCG@10: {np.mean(ndcgs):.4f}")

Hit@10: 0.1039, NDCG@10: 0.0410
